In [1]:
import os
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1" # avoiding warnings
os.environ["HUGGINGFACE_HUB_VERBOSITY"] = "error"


import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

from data import Preprocessor, create_dataloaders, GestureDataset, label_to_idx
from model import (LeNet5, create_pretrained_model, train_model,
                   make_mixup_fn, evaluate_with_tta, default_tta_transforms)


device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


## Data preprocessing & split

In [2]:
pre = Preprocessor(csv_path="../datasets/Train.csv", images_dir="../datasets/Images")
train_df, val_df, report = pre.split(return_report=True)

for key, value in report.items():
    print(f"{key}: {value}")

Removed 0 duplicate rows by img_IDS.
rows_raw: 6249
removed_duplicate_ids: 0
rows_after_dedup: 6249
rows_with_files: 6249
rows_missing_files: 0
label_distribution_raw: {'Enough/Satisfied': 695, 'Mosque': 695, 'Seat': 695, 'Temple': 694, 'Church': 694, 'Me': 694, 'Love': 694, 'You': 694, 'Friend': 694}
label_distribution_clean: {'Enough/Satisfied': 695, 'Mosque': 695, 'Seat': 695, 'Temple': 694, 'Church': 694, 'Me': 694, 'Love': 694, 'You': 694, 'Friend': 694}


## LeNet-5

In [3]:
IMAGE_SIZE_LENET = (64, 64)
BATCH_SIZE = 32

base_transform = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

train_loader, val_loader, lti = create_dataloaders(
    train_df=train_df,
    val_df=val_df,
    images_dir="../datasets/Images",
    image_size=IMAGE_SIZE_LENET,
    train_transform=base_transform,
    val_transform=base_transform,
    batch_size=BATCH_SIZE,
    num_workers=2
)

NUM_CLASSES = len(lti)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
print(f"Num classes: {NUM_CLASSES}")
print(f"Label mapping: {lti}")

Train batches: 130, Val batches: 65
Num classes: 9
Label mapping: {'Church': 0, 'Enough/Satisfied': 1, 'Friend': 2, 'Love': 3, 'Me': 4, 'Mosque': 5, 'Seat': 6, 'Temple': 7, 'You': 8}


## Train-val loop

In [4]:
lenet = LeNet5(num_classes=NUM_CLASSES, dropout=0.3)

optimizer_l = torch.optim.Adam(lenet.parameters(), lr=3e-3)
scheduler_l = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_l, max_lr=3e-3, epochs=10,
    steps_per_epoch=len(train_loader),
)

result_lenet = train_model(
    model=lenet,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=10,
    optimizer=optimizer_l,
    scheduler=scheduler_l,
    scheduler_step_per_batch=True,
    device=device
)

best_auc = max(r["val_roc_auc"] for r in result_lenet["history"])
print(f"\nBest LeNet-5 val ROC AUC: {best_auc:.4f}")

Using device: mps
Epoch   1/10 | train_loss=2.1934 | val_loss=2.1603 | val_roc_auc=0.6288
Epoch   2/10 | train_loss=2.0583 | val_loss=1.8837 | val_roc_auc=0.7659
Epoch   3/10 | train_loss=1.7866 | val_loss=1.6262 | val_roc_auc=0.8233
Epoch   4/10 | train_loss=1.5139 | val_loss=1.5053 | val_roc_auc=0.8547
Epoch   5/10 | train_loss=1.2999 | val_loss=1.4621 | val_roc_auc=0.8658
Epoch   6/10 | train_loss=1.1027 | val_loss=1.3886 | val_roc_auc=0.8759
Epoch   7/10 | train_loss=0.8992 | val_loss=1.4196 | val_roc_auc=0.8791
Epoch   8/10 | train_loss=0.7246 | val_loss=1.4692 | val_roc_auc=0.8807
Epoch   9/10 | train_loss=0.6110 | val_loss=1.5293 | val_roc_auc=0.8805
Epoch  10/10 | train_loss=0.5758 | val_loss=1.5304 | val_roc_auc=0.8810

Best LeNet-5 val ROC AUC: 0.8810


## Model tuning

In [5]:
IMAGE_SIZE_PRETRAINED = (224, 224)

pretrained_transform = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

train_loader_pt, val_loader_pt, _ = create_dataloaders(
    train_df=train_df,
    val_df=val_df,
    images_dir="../datasets/Images",
    image_size=IMAGE_SIZE_PRETRAINED,
    train_transform=pretrained_transform,
    val_transform=pretrained_transform,
    batch_size=BATCH_SIZE,
    num_workers=2
)

resnet18 = create_pretrained_model("resnet18", num_classes=NUM_CLASSES)
optimizer_r = torch.optim.Adam(resnet18.parameters(), lr=1e-4)
scheduler_r = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_r, T_max=4)

result_resnet = train_model(
    model=resnet18,
    train_loader=train_loader_pt,
    val_loader=val_loader_pt,
    epochs=4,
    optimizer=optimizer_r,
    scheduler=scheduler_r,
    device=device
)

best_auc_r = max(r["val_roc_auc"] for r in result_resnet["history"])
print(f"\nBest ResNet18 val ROC AUC: {best_auc_r:.4f}")

Using device: mps
Epoch   1/4 | train_loss=2.1205 | val_loss=1.9919 | val_roc_auc=0.8388
Epoch   2/4 | train_loss=1.7080 | val_loss=1.3481 | val_roc_auc=0.9390
Epoch   3/4 | train_loss=1.1637 | val_loss=1.0229 | val_roc_auc=0.9606
Epoch   4/4 | train_loss=0.9434 | val_loss=0.9572 | val_roc_auc=0.9650

Best ResNet18 val ROC AUC: 0.9650


## Augmentations

In [6]:
aug_train_transform = A.Compose([
    A.Affine(translate_percent=0.05, scale=(0.9, 1.1), rotate=(-15, 15), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
    A.GaussNoise(std_range=(0.01, 0.05), p=0.2),
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(8, 16), hole_width_range=(8, 16), p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

train_loader_aug, val_loader_aug, _ = create_dataloaders(
    train_df=train_df,
    val_df=val_df,
    images_dir="../datasets/Images",
    image_size=IMAGE_SIZE_PRETRAINED,
    train_transform=aug_train_transform,
    val_transform=pretrained_transform,
    batch_size=BATCH_SIZE,
    num_workers=2
)

In [7]:
resnet18_aug = create_pretrained_model("resnet18", num_classes=NUM_CLASSES)
optimizer_a = torch.optim.Adam(resnet18_aug.parameters(), lr=1e-4)
scheduler_a = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_a, T_max=4)

result_aug = train_model(
    model=resnet18_aug,
    train_loader=train_loader_aug,
    val_loader=val_loader_aug,
    epochs=4,
    optimizer=optimizer_a,
    scheduler=scheduler_a,
    device=device
)

best_auc_aug = max(r["val_roc_auc"] for r in result_aug["history"])
print(f"\nBest ResNet18 + augmentations val ROC AUC: {best_auc_aug:.4f}")
print(f"Without augmentations: {best_auc_r:.4f}")
print(f"Gain: {best_auc_aug - best_auc_r:+.4f}")

Using device: mps
Epoch   1/4 | train_loss=2.1545 | val_loss=2.0451 | val_roc_auc=0.7901
Epoch   2/4 | train_loss=1.8358 | val_loss=1.4450 | val_roc_auc=0.9280
Epoch   3/4 | train_loss=1.3151 | val_loss=1.0701 | val_roc_auc=0.9566
Epoch   4/4 | train_loss=1.0845 | val_loss=0.9834 | val_roc_auc=0.9625

Best ResNet18 + augmentations val ROC AUC: 0.9625
Without augmentations: 0.9650
Gain: -0.0025


## MixUp & CutMix

In [8]:
mixup_fn = make_mixup_fn(num_classes=NUM_CLASSES, alpha=0.4, mode="mixup")

resnet18_mix = create_pretrained_model("resnet18", num_classes=NUM_CLASSES)
optimizer_m = torch.optim.Adam(resnet18_mix.parameters(), lr=1e-4)
scheduler_m = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_m, T_max=10)

result_mix = train_model(
    model=resnet18_mix,
    train_loader=train_loader_aug,
    val_loader=val_loader_aug,
    epochs=10,
    optimizer=optimizer_m,
    scheduler=scheduler_m,
    mixup_fn=mixup_fn,
    device=device
)

best_auc_mix = max(r["val_roc_auc"] for r in result_mix["history"])
print(f"\nResNet18 + aug + MixUp val ROC AUC: {best_auc_mix:.4f}")

Using device: mps
Epoch   1/10 | train_loss=2.1654 | val_loss=2.0882 | val_roc_auc=0.7827
Epoch   2/10 | train_loss=1.9558 | val_loss=1.5217 | val_roc_auc=0.9267
Epoch   3/10 | train_loss=1.4892 | val_loss=0.9344 | val_roc_auc=0.9656
Epoch   4/10 | train_loss=1.2698 | val_loss=0.7290 | val_roc_auc=0.9783
Epoch   5/10 | train_loss=1.1002 | val_loss=0.6149 | val_roc_auc=0.9841
Epoch   6/10 | train_loss=1.0644 | val_loss=0.5814 | val_roc_auc=0.9861
Epoch   7/10 | train_loss=1.0322 | val_loss=0.5459 | val_roc_auc=0.9873
Epoch   8/10 | train_loss=1.0176 | val_loss=0.5493 | val_roc_auc=0.9875
Epoch   9/10 | train_loss=1.0119 | val_loss=0.5494 | val_roc_auc=0.9879
Epoch  10/10 | train_loss=0.9865 | val_loss=0.5611 | val_roc_auc=0.9876

ResNet18 + aug + MixUp val ROC AUC: 0.9879


In [9]:
cutmix_fn = make_mixup_fn(num_classes=NUM_CLASSES, alpha=1.0, mode="cutmix")

resnet18_cut = create_pretrained_model("resnet18", num_classes=NUM_CLASSES)
optimizer_c = torch.optim.Adam(resnet18_cut.parameters(), lr=1e-4)
scheduler_c = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_c, T_max=10)

result_cut = train_model(
    model=resnet18_cut,
    train_loader=train_loader_aug,
    val_loader=val_loader_aug,
    epochs=10,
    optimizer=optimizer_c,
    scheduler=scheduler_c,
    mixup_fn=cutmix_fn,
    device=device
)

best_auc_cut = max(r["val_roc_auc"] for r in result_cut["history"])
print(f"\nResNet18 + aug + CutMix val ROC AUC: {best_auc_cut:.4f}")

Using device: mps
Epoch   1/10 | train_loss=2.1817 | val_loss=2.1208 | val_roc_auc=0.7255
Epoch   2/10 | train_loss=2.0839 | val_loss=1.8756 | val_roc_auc=0.8847
Epoch   3/10 | train_loss=1.8719 | val_loss=1.3859 | val_roc_auc=0.9398
Epoch   4/10 | train_loss=1.6783 | val_loss=1.0665 | val_roc_auc=0.9627
Epoch   5/10 | train_loss=1.5154 | val_loss=0.8646 | val_roc_auc=0.9737
Epoch   6/10 | train_loss=1.4768 | val_loss=0.7501 | val_roc_auc=0.9790
Epoch   7/10 | train_loss=1.3615 | val_loss=0.7064 | val_roc_auc=0.9813
Epoch   8/10 | train_loss=1.4023 | val_loss=0.6678 | val_roc_auc=0.9827
Epoch   9/10 | train_loss=1.3577 | val_loss=0.6892 | val_roc_auc=0.9827
Epoch  10/10 | train_loss=1.3478 | val_loss=0.6314 | val_roc_auc=0.9836

ResNet18 + aug + CutMix val ROC AUC: 0.9836


## Test-Time Augmentation

In [10]:
all_labels = set(train_df["Label"].unique()) | set(val_df["Label"].unique())
lti = label_to_idx(all_labels)

val_ds_raw = GestureDataset(
    val_df,
    images_dir="../datasets/Images",
    label_to_idx=lti,
    image_size=IMAGE_SIZE_PRETRAINED,
    transform=None
)

best_model = result_mix["model"]

tta_transforms = default_tta_transforms(IMAGE_SIZE_PRETRAINED)

tta_result = evaluate_with_tta(
    best_model, val_ds_raw, tta_transforms, device
)

print(f"Standard ROC AUC (best model): {best_auc_mix:.4f}")
print(f"TTA ROC AUC:                   {tta_result['roc_auc']:.4f}")
print(f"Gain from TTA:                 {tta_result['roc_auc'] - best_auc_mix:+.4f}")

Standard ROC AUC (best model): 0.9879
TTA ROC AUC:                   0.9876
Gain from TTA:                 -0.0003


In [11]:
print("=" * 55)
print(f"{'Experiment':<35} {'ROC AUC':>10}")
print("-" * 55)
print(f"{'LeNet-5 (baseline)':<35} {best_auc:>10.4f}")
print(f"{'ResNet18 (no aug)':<35} {best_auc_r:>10.4f}")
print(f"{'ResNet18 + augmentations':<35} {best_auc_aug:>10.4f}")
print(f"{'ResNet18 + aug + MixUp':<35} {best_auc_mix:>10.4f}")
print(f"{'ResNet18 + aug + CutMix':<35} {best_auc_cut:>10.4f}")
print(f"{'ResNet18 + aug + TTA':<35} {tta_result['roc_auc']:>10.4f}")
print("=" * 55)

Experiment                             ROC AUC
-------------------------------------------------------
LeNet-5 (baseline)                      0.8810
ResNet18 (no aug)                       0.9650
ResNet18 + augmentations                0.9625
ResNet18 + aug + MixUp                  0.9879
ResNet18 + aug + CutMix                 0.9836
ResNet18 + aug + TTA                    0.9876
